<a href="https://colab.research.google.com/github/Kadosh-Tech33/A3/blob/main/C%C3%B3pia_de_Lab02_Aula05_Engenharia_Atributos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratório 02 — Estruturas de dados e engenharia de características
### Aprendizado de Máquina Supervisionado — Curso de Inteligência Artificial
**FMU — 2026.2** · Prof. Luciano Tadeu Pereira · Aula 05 — 16/09/2026

---
Aqui você transforma colunas brutas em **atributos que carregam sinal**. Saída: `base_credito_features.csv`. Regra de ouro da aula: todo atributo novo precisa de uma justificativa de negócio, não só de um ganho de métrica.

> **Como usar:** rode as células **em ordem**. Onde houver `# SUA VEZ`, escreva o código antes de avançar.
> Onde houver **Responda:**, escreva a resposta na própria célula de texto.

***ATENÇÃO:*** Antes de começar, faça uma cópia deste notebook para sua conta do Google Drive. Assim, você poderá editar e testar sem modificar o original.

Para isso, clique em "Arquivo" → "Salvar uma cópia no Drive".

OBS: Para executar este notebook, não é necessário um ambiente de execução com GPU.


## 1. As quatro estruturas que você vai usar o semestre inteiro

In [2]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

lista  = [3200.0, 8100.0, 1500.0]                 # heterogênea, lenta, flexível
vetor  = np.array(lista)                          # homogênea, vetorizada, rápida
serie  = pd.Series(lista, index=["Ana","Beto","Caio"], name="renda")
tabela = pd.DataFrame({"renda": lista, "atrasos": [2, 0, 5]}, index=["Ana","Beto","Caio"])

print("lista  * 2 ->", lista * 2)          # repete a lista!
print("vetor  * 2 ->", vetor * 2)          # multiplica os valores
print("\nsérie acima da média:\n", serie[serie > serie.mean()].to_string())
display(tabela.assign(risco=lambda t: np.where(t.atrasos >= 2, "alto", "baixo")))

lista  * 2 -> [3200.0, 8100.0, 1500.0, 3200.0, 8100.0, 1500.0]
vetor  * 2 -> [ 6400. 16200.  3000.]

série acima da média:
 Beto    8100.0


,renda,atrasos,risco
Ana,3200.0,2,alto
Beto,8100.0,0,baixo
Caio,1500.0,5,alto


**Responda:** por que `lista * 2` e `vetor * 2` dão resultados diferentes? Em qual das duas você faria contas com 3 milhões de linhas?\n\

a lista 2 é uma repetição de numeros ja a lista do vetor é uma mutiplicação então nesse caso teria que ser a lista vetor, pois pelo o fato de ser uma opereção vetorizada, ele processa os dados muito mais rapido em relação a lista 2.

## 2. Carga e recorte\n\n`.loc` usa rótulos, `.iloc` usa posições. Trocar os dois é o erro nº 1 de quem está começando.

In [3]:
df = pd.read_csv("base_credito_limpa.csv")

print(df.loc[0, ["id_cliente","renda_mensal","score_bureau"]].to_dict())   # rótulos
print(df.iloc[0, :3].to_dict())                                            # posições

risco = df.loc[(df.parcelas_atraso_12m >= 2) & (df.score_bureau < 500)]
print(f"\nclientes com 2+ atrasos e score < 500: {len(risco)} "
      f"| inadimplência do grupo: {risco.inadimplente.mean():.1%} "
      f"| da base: {df.inadimplente.mean():.1%}")

{'id_cliente': 11194, 'renda_mensal': 4830.65, 'score_bureau': 433.0}
{'id_cliente': 11194, 'idade': 46.0, 'renda_mensal': 4830.65}

clientes com 2+ atrasos e score < 500: 116 | inadimplência do grupo: 35.3% | da base: 17.9%


## 3. Atributos derivados: onde mora o ganho\n\nUm atributo bem construído costuma valer mais que trocar de algoritmo. Os três primeiros abaixo são padrão de mercado em risco de crédito.

In [4]:
df["parcela_estimada"]      = (df.valor_emprestimo / df.prazo_meses).round(2)
df["comprometimento_renda"] = (df.parcela_estimada / df.renda_mensal).round(4)
df["emprestimo_sobre_renda"]= (df.valor_emprestimo / df.renda_mensal).round(3)
df["anos_emprego"]          = (df.tempo_emprego_meses / 12).round(2)

novos = ["parcela_estimada","comprometimento_renda","emprestimo_sobre_renda","anos_emprego"]
display(df[novos].describe().round(3))

print("\ncorrelação com o alvo (Pearson):")
print(df[novos + ["score_bureau","renda_mensal","inadimplente"]]
      .corr()["inadimplente"].drop("inadimplente").round(3).sort_values().to_string())

,parcela_estimada,comprometimento_renda,emprestimo_sobre_renda,anos_emprego
count,3000.000,2861.000,2861.000,3000.000
mean,690.667,0.212,3.372,3.306
std,798.592,0.183,1.527,2.184
min,21.870,0.004,0.127,0.080
25%,237.715,0.091,2.059,1.670
50%,448.730,0.157,3.385,2.830
75%,837.388,0.266,4.673,4.420
max,10065.320,0.997,5.997,21.830



correlação com o alvo (Pearson):
score_bureau             -0.216
anos_emprego             -0.034
renda_mensal             -0.003
emprestimo_sobre_renda    0.124
parcela_estimada          0.172
comprometimento_renda     0.230


**Responda:** `renda_mensal` sozinha tem correlação quase nula com o alvo, mas `comprometimento_renda` não. Explique por que a razão informa mais que o valor absoluto.\n\n

a renda mensal ela é um dado "solto" por que a pessoa pode receber um valor alto e mesmo assim esta passando por um sufoco financeiro, ja o comprometimento com a renda ele é um dado real, por que mostra o compromisso financeiro em relação a capacidade de pagamento que aquela pessoa tem.

## 4. Discretização (binning) e taxa por faixa\n\nBinning perde informação — e ganha legibilidade. Em crédito, a faixa é o que vai para a política.

In [ ]:
df["faixa_score"] = pd.cut(df.score_bureau, bins=[-1, 400, 550, 700, 1001],
                           labels=["muito_baixo","baixo","medio","alto"])

tab = (df.groupby("faixa_score", observed=True)
         .agg(clientes=("id_cliente","count"), inadimplencia=("inadimplente","mean")).round(3))
tab["lift"] = (tab.inadimplencia / df.inadimplente.mean()).round(2)
display(tab)

ax = tab.inadimplencia.plot(kind="bar", color="#ED0203", figsize=(7, 3.5),
                            title="Inadimplência por faixa de score")
ax.axhline(df.inadimplente.mean(), color="#3A0604", ls="--", label="média da base")
ax.legend(); plt.tight_layout(); plt.show()

## 5. Codificação de categóricas — e a armadilha do Label Encoder

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

amostra = df[["tipo_moradia"]].head(6)
ordinal = OrdinalEncoder().fit_transform(amostra)          # inventa uma ordem!
onehot  = OneHotEncoder(sparse_output=False).fit_transform(amostra)

print("categorias:", amostra.tipo_moradia.tolist())
print("ordinal   :", ordinal.ravel().tolist(), "  <- sugere que 'financiada' > 'alugada'")
print("one-hot   :", onehot.shape, "colunas, sem ordem implícita")

print("\ncardinalidade das categóricas:")
print(df[["tipo_moradia","finalidade","uf"]].nunique().to_string())

**Responda:** em qual situação o `OrdinalEncoder` seria a escolha *correta*? Dê um exemplo de coluna desta base ou de outra.\n\n

o ordinalColder ele é bom quando tem um nivel de herearquia de valores, exemplo o tamanho de roupas P, M, G, GG. O M, é maior que o P e vem depois dele, o G é maior que O M e vem depois dele, ou seja, tem uma horden de hirearqui de valores.


## 6. Vazamento de dados: o erro que só aparece em produção\n\nPadronizar ou imputar **antes** de separar treino e teste faz a estatística do teste vazar para o treino. O resultado é um número bonito que não se repete no mundo real.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[["renda_mensal","comprometimento_renda"]].fillna(df[["renda_mensal","comprometimento_renda"]].median())
y = df.inadimplente

# ERRADO: ajusta o scaler na base inteira
X_errado = StandardScaler().fit_transform(X)
a_tr, a_te, *_ = train_test_split(X_errado, y, test_size=.25, stratify=y, random_state=42)

# CERTO: separa primeiro, ajusta só no treino
b_tr, b_te, *_ = train_test_split(X, y, test_size=.25, stratify=y, random_state=42)
sc = StandardScaler().fit(b_tr)

print("média do teste sob o scaler global :", np.round(a_te.mean(axis=0), 4))
print("média do teste sob o scaler correto:", np.round(sc.transform(b_te).mean(axis=0), 4))
print("\nNo caso correto a média do teste NÃO é zero — e é assim que tem de ser:")
print("o modelo em produção nunca viu os dados que vai receber.")

média do teste sob o scaler global : [-0.0267  0.0033]
média do teste sob o scaler correto: [-0.0348  0.0045]

No caso correto a média do teste NÃO é zero — e é assim que tem de ser:
o modelo em produção nunca viu os dados que vai receber.


## 7. Salvar a base com atributos

In [6]:
df.to_csv("base_credito_features.csv", index=False)
print("salvo:", df.shape, "| colunas novas:", novos + ["faixa_score"])

from google.colab import files      # comente se não estiver no Colab
files.download("base_credito_features.csv")

salvo: (3000, 18) | colunas novas: ['parcela_estimada', 'comprometimento_renda', 'emprestimo_sobre_renda', 'anos_emprego', 'faixa_score']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. SUA VEZ\n\nCrie **dois** atributos novos e defenda cada um em uma frase de negócio.

In [15]:
# mostre o dinheiro que sobre no mes
df["renda_disponivel"] = df.renda_mensal - df.parcela_estimada
# mostra o valor que sobra apos pagar a parcela
print(df)

# atributo 2: estabilidade no trabalho em relação ao contrato
df["estabilidade_vs_prazo"] = (df.emprego_meses / df.prazo_meses).round(2)
# confere se a estabilidade profissional do cliente
(df)

      id_cliente  idade  renda_mensal  tempo_emprego_meses  valor_emprestimo  \
0          11194   46.0       4830.65                 20.0           6823.17   
1          11574   38.0       4290.02                 55.0          21556.28   
2          10007   39.0       6167.22                 24.0           6167.69   
3          10004   48.0       1546.41                  5.0           3300.10   
4          10925   36.0       2358.35                  7.0          12878.37   
...          ...    ...           ...                  ...               ...   
2995       12764   48.0       4974.97                 52.0          16559.43   
2996       10906   61.0       3619.03                 59.0           4904.15   
2997       11097   55.0       3231.66                 20.0          17514.03   
2998       10236   31.0       4276.05                 47.0          18598.89   
2999       11062   54.0       9951.86                 29.0          40105.58   

      prazo_meses  parcelas_atraso_12m 

---\n## Entrega\n\nPoste o link no AVA até **23/09**, com a tabela de faixas, as respostas e os dois atributos autorais. Conta como atividade de N1.